# Investment Research Agent: Multi-Agent Equity Research & Factual Grounding

---

### Academic Information & Course Attribution
* **Institution:** University of San Diego (USD), Shiley-Marcos School of Engineering
* **Degree Program:** Master of Science in Applied Artificial Intelligence (MS-AAI)
* **Course:** **AAI-520-IN2: Natural Language Processing and Generative AI**
* **Instructor:** **Prof. Rajesh Soni, M.S.**
* **Project Repository:** `investment-research-agent`
* **Project Team:**
  1. **Sai Krishna Akula** (`nakula@sandiego.edu`)
  2. **Ashok Kumar Bhairwal** (`abhairwal@sandiego.edu`)
  3. **Diaesh Antony** (`dantony@sandiego.edu`)

---

## 1. Executive Summary & System Architecture

### The Financial Research Challenge
Equity research requires gathering, analyzing, and synthesizing high-volume, heterogeneous financial information:
- **SEC 10-K & 10-Q Filings:** Lengthy legal text, financial statement footnotes, and risk factor disclosures.
- **Fundamental & Valuation Metrics:** Earnings reports, quarterly revenues, profit margins, EBITDA, and forward multiples.
- **Market Price & Momentum Signals:** OHLCV price histories, 50-day and 200-day moving averages (SMAs), 14-day RSI, and beta.
- **Market News Catalysts & Sentiment:** Breaking developments, corporate partnerships, regulatory actions, and macro interest rates.

Producing a complete equity report manually requires several hours of an analyst's time.

### Failure Modes of Single-Prompt LLM Systems
Attempting to automate equity research using a single LLM prompt or a simple linear chain exhibits four well-documented failure modes:
1. **Parametric Hallucination:** Language models generate plausible next tokens based on static weights. When queried for specific financial ratios or quarterly revenues, models frequently hallucinate realistic-looking figures.
2. **Context Degradation ("Lost-in-the-Middle"):** Supplying extensive SEC filings in a single context window degrades attention, causing the model to omit critical balance sheet details or risk disclosures.
3. **Knowledge Cutoffs:** Static weights lack access to current stock quotes, recent regulatory filings, and live news.
4. **Absence of Critical Audit:** Monolithic prompts lack an independent verification step to check whether stated figures match authoritative financial sources.

---

### System Architecture Overview
To address these limitations, this system implements an autonomous multi-agent orchestration architecture. Instead of relying on a monolithic prompt, the research workflow is divided across specialized components coordinated through a shared blackboard state (`GraphState`).

The framework is organized around eight core components:
1. **Agents:** Programs that use a language model for reasoning, executing an iterative loop (Reason $\to$ Act $\to$ Observe) with defined roles and tool access.
2. **Tools:** Standard Python functions that query external data sources (Yahoo Finance, SEC EDGAR, Alpha Vantage). Tools supply deterministic data so the language model does not guess numbers.
3. **State (`GraphState`):** A shared dictionary (blackboard) passed from node to node. It holds the user's question, the target ticker, findings from each specialist, raw tool results, and review scores.
4. **Memory (`GraphMemory` & `SessionStore`):**
   - *Short-Term Memory (`GraphMemory`)*: Stores execution state checkpoints during a single research run.
   - *Long-Term Memory (`SessionStore`)*: Uses an SQLite database to store multi-turn conversation history and cached reports, allowing users to ask follow-up questions without re-running data queries.
5. **StateGraph:** An execution graph that coordinates nodes (agents or Python functions) along defined paths from `Start` to `End`.
6. **Conditional Routers:** State-aware decision rules that direct workflow branches based on query intent (fast-path conceptual answers vs. deep research) and audit outcomes (pass vs. revise).
7. **Review Loop (Critic & Refiner):** A two-stage verification gate where a Critic checks every number in the draft against raw tool data, and a Refiner corrects flagged sections.
8. **Progress Tracking & Trajectory:** Real-time console streaming showing active agents (`Agent: [...]`) and tool calls (`Tool: [...]`), backed by a full step-by-step trajectory audit table.


In [1]:
import sys
from pathlib import Path

# Resolve project src directory
project_root = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "src").exists():
        project_root = candidate
        break
sys.path.insert(0, str(project_root / "src"))

from research_agent.agent_framework import (
    END,
    Agent,
    GraphMemory,
    GraphState,
    StateGraph,
    Tool,
)
from research_agent.interfaces import get_llm

llm = get_llm()
print("Framework imported successfully. LLM client initialized.")

Framework imported successfully. LLM client initialized.


## 2. Core Concept 1: What is an Agent?

### Theoretical Foundation: Perception-Action and the ReAct Paradigm
In artificial intelligence theory (Russell & Norvig, *Artificial Intelligence: A Modern Approach*), an **Agent** is defined as an entity that perceives its environment through sensors (inputs) and acts upon that environment through effectors (tools).

An autoregressive language model alone computes token probabilities $P(w_t \mid w_{<t})$ in a feed-forward manner. It lacks sensory access to live data and cannot independently verify or correct its output.

To impart agency, this framework implements the **ReAct (Reasoning and Acting)** framework (Yao et al., 2022). In ReAct, the model alternates between generating reasoning thoughts and emitting structured actions:
1. **Thought:** The model generates an internal rationale evaluating the current state.
2. **Action:** The model emits a structured tool invocation with arguments (e.g. `lookup_ticker(query='NVDA')`).
3. **Observation:** The host environment executes the tool and appends the result to the conversation context.
4. **Reflection:** The model processes the observation, deciding whether to call another tool or generate the final answer.

```
       ┌────────────────────────────────────────────────────────┐
       │                 THE AGENT ReAct LOOP                   │
       │                                                        │
       │  Prompt ───▶ [Reason / Thought]                        │
       │                     │                                  │
       │                     ▼                                  │
       │             Tool Call Required?                        │
       │                 /        \                             │
       │         [YES]  /          \  [NO]                      │
       │               ▼            ▼                           │
       │        [Tool Action]   [Final Answer Formulated]       │
       │               │                                        │
       │               ▼                                        │
       │        [Observation] ───▶ (Append to Context & Repeat) │
       └────────────────────────────────────────────────────────┘
```

The execution loop terminates when the model produces an answer without requesting tools, or when the bounded iteration limit ($T_{\max}$) is reached, preventing infinite execution loops.

### Agent Implementation in this Framework
An `Agent` instance encapsulates:
- `name`: Identifier used in execution logs and telemetry.
- `role`: System prompt establishing the persona and operational constraints.
- `tools`: A dictionary of callable `Tool` instances.
- `llm`: The language model client.
- `max_iterations`: Upper bound on reasoning-acting cycles.
- `run(prompt, on_progress=None)`: The primary method executing the ReAct loop.
- `as_tool()`: Wraps the agent as a callable function so an orchestrator agent can delegate sub-tasks to it.


In [2]:
# Demonstrate Concept 1: Creating an Agent with a deterministic calculation tool
def calculate_financial_health_score(operating_margin: float, debt_to_equity: float) -> dict:
    """Calculates a composite financial health score from operating margin and leverage."""
    score = (operating_margin * 0.7) - (debt_to_equity * 20.0)
    verdict = "Strong" if score > 30 else "Moderate" if score > 10 else "High Risk"
    return {"composite_score": round(score, 2), "health_verdict": verdict}


health_tool = Tool(
    name="calculate_financial_health_score",
    description="Calculate composite financial health score from margin and debt ratio.",
    func=calculate_financial_health_score,
)

analyst_agent = Agent(
    name="FinancialHealthSpecialist",
    role="Evaluate company balance sheet resilience using the health score tool.",
    llm=llm,
    tools=[health_tool],
    max_iterations=4,
)

query = "Evaluate NVDA financial health if operating margin is 62.0% and debt-to-equity is 0.42."
response = analyst_agent.run(query)

print(f"Agent Name     : {analyst_agent.name}")
print(f"Iterations Run : {response.iterations}")
print(f"Tools Used     : {[t.name for t in analyst_agent.tools.values()]}")
print(f"Trace Steps    : {[step.step_type for step in response.trace]}")
print(f"\n--- AGENT FINAL ANSWER ---\n{response.content}")

Agent Name     : FinancialHealthSpecialist
Iterations Run : 2
Tools Used     : ['calculate_financial_health_score']
Trace Steps    : ['tool_call', 'tool_result', 'thought', 'final_answer']

--- AGENT FINAL ANSWER ---
**Financial Health Evaluation for NVIDIA (NVDA)**

- **Operating Margin:** 62.0%
- **Debt‑to‑Equity Ratio:** 0.42

Using the provided **calculate_financial_health_score** tool, the composite financial health score is **35.0**, leading to a **“Strong”** health verdict. This indicates that, given the high operating margin and modest leverage, NVIDIA is in a solid financial position.


## 3. Core Concept 2: What are Tools?

### Theoretical Foundation: Tool-Augmented Language Models
In modern natural language processing (Schick et al., 2023, *Toolformer*; Mialon et al., 2023, *Augmented Language Models*), tools bridge the gap between parametric memory and non-parametric computation:
- **Parametric capabilities:** Natural language comprehension, synthesis, and qualitative drafting.
- **Non-parametric capabilities:** Deterministic arithmetic, live database querying, and legal text retrieval.

A language model cannot reliably perform 10-digit floating point arithmetic or look up yesterday's closing stock price from weights. By binding Python functions via formal JSON schemas, the language model acts as a controller that decides *when* and *with what arguments* to invoke an external function.

### Safety and Error Resilience
The `Tool` class provides execution sandboxing:
1. **Schema Validation:** Function arguments are parsed and mapped to the underlying Python signature.
2. **Exception Handling:** If an API call fails or encounters a network error, the exception is caught and returned as an informative error string. This allows the calling model to observe the failure, adjust its parameters, and retry rather than crashing the workflow.

### Overview of Financial Tools
| Tool Name | Source | Purpose | Input Parameter | Output Format |
| :--- | :--- | :--- | :--- | :--- |
| `lookup_ticker` | Yahoo Finance | Resolves company names to ticker symbols | `query: str` | `{"ticker": str, "company": str}` |
| `get_earnings_data` | Yahoo Finance / SEC | Extracts revenue, margins, cash flow, EPS | `ticker: str` | Dict of financial metrics |
| `search_sec_filings` | SEC EDGAR | Searches 10-K/10-Q filing disclosures | `ticker: str`, `query: str` | Relevant filing excerpts |
| `get_market_technicals` | Yahoo Finance | Computes 50/200 SMAs, 14-day RSI, beta | `ticker: str` | Dict of technical metrics |
| `get_macro_indicators` | Alpha Vantage / Fed | Retrieves Fed Funds Rate, Treasury yield | None | Dict of macro indicators |
| `get_news_data` | Yahoo Finance / RSS | Collects recent news and calculates sentiment | `ticker: str`, `days: int` | Dict of articles and sentiment |


In [3]:
# Demonstrate Concept 2: Deterministic Tool Execution
from research_agent.tools.filings import search_sec_filings
from research_agent.tools.yahoo_finance import lookup_ticker

# 1. Symbol Resolution Tool
ticker_info = lookup_ticker("Apple")
print("1. Tool Execution: lookup_ticker('Apple')")
print(f"   Result: {ticker_info}")

# 2. Regulatory Filing Retrieval Tool
filing_excerpt = search_sec_filings("NVDA", query="risk factors competition", max_results=1)
print("\n2. Tool Execution: search_sec_filings('NVDA', query='risk factors competition')")
print(f"   Excerpt Preview: {filing_excerpt[:220]}...")

1. Tool Execution: lookup_ticker('Apple')
   Result: {'ticker': 'AAPL', 'company': 'Apple Inc.', 'source': 'yahoo_finance'}

2. Tool Execution: search_sec_filings('NVDA', query='risk factors competition')
   Excerpt Preview: [Passage 1 | Form 10-K | Item 1A. Risk Factors] Our market is intensely competitive and characterized by rapid technological change. We face intense competition from established semiconductor companies and hyperscale cloud providers...


## 4. Core Concept 3: What is State? The Blackboard Architecture (`GraphState`)

### Theoretical Foundation: The Blackboard Architectural Pattern
In classical distributed artificial intelligence (Erman et al., 1980, *The Hearsay-II Speech-Understanding System*; Buschmann et al., 1996), the **Blackboard Pattern** solves problems where multiple specialized sources of expertise must collaborate on a complex problem without direct point-to-point coupling.

```
       ┌────────────────────────────────────────────────────────┐
       │               SHARED BLACKBOARD (GraphState)           │
       │                                                        │
       │  • question     : "Analyze NVIDIA fundamentals..."     │
       │  • active_ticker: "NVDA"                              │
       │  • agents       : {EarningsAnalyst: ..., News: ...}   │
       │  • tools        : {get_earnings_data: {rev: 30.04B}}  │
       │  • review       : {score: 9.5, passed: True}          │
       │  • trajectory   : [Step 1, Step 2, Step 3, ...]       │
       │  • output       : "Final Institutional Brief..."      │
       └────────────────────────────┬───────────────────────────┘
               ▲                   ▲                   ▲
               │                   │                   │
               ▼                   ▼                   ▼
        EarningsAnalyst     MarketAnalyst         Critic/Refiner
        (reads/writes)      (reads/writes)        (audits blackboard)
```

### Linear Chaining vs. Blackboard State
In a traditional linear prompt chain ($A \to B \to C$):
- Each step concatenates previous text outputs, resulting in $O(N)$ prompt growth.
- Factual precision degrades as intermediate agents summarize and re-summarize numeric findings.
- An auditing component cannot access the original raw tool observations because they were lost during summarization.

In this framework's Blackboard architecture:
- All agents read from and write to a single structured object: `GraphState`.
- Raw tool observations are preserved in `state.tools`, enabling the Critic to verify draft assertions against primary source data.
- The execution history is preserved in `state.trajectory` for telemetry and debugging.


In [4]:
# Demonstrate Concept 3: Shared Blackboard State Management
state = GraphState(question="Analyze NVIDIA Q2 fundamentals and margins")
state.active_ticker = "NVDA"

# EarningsAnalyst posts findings and raw tool observations to blackboard
state.agents["EarningsAnalyst"] = "Q2 Revenue: $30.04B (+122% YoY), Gross Margin: 75.1%."
state.tools["get_earnings_data"] = {"revenue_b": 30.04, "gross_margin_pct": 75.1, "ebitda_b": 18.64}

# MarketAnalyst posts technical metrics to blackboard
state.agents["MarketAnalyst"] = "Current Price: $128.50, 50-day SMA: $121.30, 14-day RSI: 63.34."
state.tools["get_market_technicals"] = {"price": 128.50, "sma_50": 121.30, "rsi_14": 63.34}

print(f"Active Ticker        : {state.active_ticker}")
print(f"Agents on Blackboard : {list(state.agents.keys())}")
print(f"Tools Preserved      : {list(state.tools.keys())}")
print(f"Blackboard Schema    : {list(state.to_dict().keys())}")

Active Ticker        : NVDA
Agents on Blackboard : ['EarningsAnalyst', 'MarketAnalyst']
Tools Preserved      : ['get_earnings_data', 'get_market_technicals']
Blackboard Schema    : ['question', 'output', 'state_unique_id', 'thread_id', 'agents', 'tools', 'trajectory', 'refine_count', 'review', 'active_ticker']


## 5. Core Concept 4: What is Memory? Dual-Horizon Architecture

### Theoretical Foundation: Working Memory vs. Episodic Memory
Drawing from cognitive architectures (Baddeley, 1992; Newell's SOAR), agentic memory is partitioned into two operational horizons:

```
┌────────────────────────────────────────────────────────────────────────┐
│                          MEMORY ARCHITECTURE                           │
├───────────────────────────────────┬────────────────────────────────────┤
│ 1. Transient Memory (GraphMemory) │ 2. Persistent Memory (SessionStore)│
├───────────────────────────────────┼────────────────────────────────────┤
│ • Horizon: Single workflow run    │ • Horizon: Multi-turn chat session │
│ • Storage: In-memory / JSON diffs │ • Storage: SQLite Relational DB    │
│ • Contents: Intermediate traces,  │ • Contents: Message turns, rolling │
│   blackboard updates, node deltas │   summaries, TTL cached briefs     │
│ • Purpose: Agent coordination &   │ • Purpose: Multi-turn conversation │
│   execution telemetry auditing    │   continuity & entity recall       │
└───────────────────────────────────┴────────────────────────────────────┘
```

1. **Transient Working Memory (`GraphMemory`):**
   - Scoped to an individual workflow run.
   - Captures state checkpoints after each node execution, providing persistence for debugging and run recovery.
2. **Persistent Conversational Memory (`SessionStore`):**
   - Scoped to a continuous multi-turn dialogue (`thread_id`).
   - Persisted in an SQLite relational database (`data/memory/sessions.sqlite`).
   - Stores user messages, assistant responses, rolling summaries, and Time-To-Live (TTL) cached research briefs.

When a user asks a follow-up question (e.g. *"What were the debt risks for the stock we just analyzed?"*), the session resolves the active entity from conversation memory and answers directly from the cached brief, avoiding redundant tool calls and API latency.


In [5]:
# Demonstrate Concept 4: State Checkpointing and Memory Retrieval
memory = GraphMemory()
checkpoint_id = memory.write(state)

print(f"Checkpoint ID Generated: {checkpoint_id}")
print(f"Memory Checkpoints     : {memory.list_state_ids()}")

# Restore state from transient memory
restored = memory.read(checkpoint_id)
print(f"Restored Ticker        : {restored.active_ticker}")
print(f"Restored Finding       : {restored.agents['EarningsAnalyst']}")

Checkpoint ID Generated: state_20261012_235130_25adad
Memory Checkpoints     : ['state_20261012_235130_25adad']
Restored Ticker        : NVDA
Restored Finding       : Q2 Revenue: $30.04B (+122% YoY), Gross Margin: 75.1%.


## 6. Core Concept 5: StateGraph Workflow Engine & Architecture Topology

### Theoretical Foundation: Cyclic State Graphs vs. Linear DAGs
Standard pipeline orchestrators (such as Apache Airflow or basic LangChain chains) enforce Directed Acyclic Graphs (DAGs). While DAGs are well-suited for linear ETL jobs, they cannot represent iterative agentic workflows that require:
- Cyclic refinement loops (re-evaluating a draft until it satisfies quality criteria).
- State-dependent conditional branching (diverting traffic based on intermediate classifications).

`StateGraph` represents a workflow as a graph $G = (V, E)$, where:
- **Vertices ($V$):** Computational nodes that execute either an `Agent`, a parallel group of agents, or a Python function.
- **Edges ($E$):** Transitions that route the state from one node to the next.

---

### Architecture Topology Diagram
The workflow graph structure is illustrated below. The high-resolution diagram rendered by the framework is saved in [`notebooks/architecture_topology.png`](architecture_topology.png):

![Workflow Architecture Topology Diagram](architecture_topology.png)

```mermaid
flowchart TD
    __start__([Start]) --> planner
    planner["Agent: ResearchPlanner<br/>(Intent Classifier & Symbol Resolver)<br/>Tool: lookup_ticker"]
    planner -.->|RESEARCH_REQUIRED| orchestrator
    planner -.->|Direct Concept Answer| __end__([End])

    subgraph Specialists ["Specialist Sub-Agents & Financial Data Tools (Sequential Parallel Execution)"]
        earningsanalyst["EarningsAnalyst<br/>Tool: get_earnings_data, search_sec_filings"]
        marketanalyst["MarketAnalyst<br/>Tool: get_market_technicals, get_macro_indicators"]
        newsanalyst["NewsAnalyst<br/>Tool: get_news_data (5-Stage Pipeline)"]
    end

    orchestrator["Agent: ResearchRouter<br/>(Lead Orchestrator & Dispatcher)"]
    orchestrator -.->|delegate_to_earningsanalyst| earningsanalyst
    orchestrator -.->|delegate_to_marketanalyst| marketanalyst
    orchestrator -.->|delegate_to_newsanalyst| newsanalyst

    orchestrator --> draft_writer
    draft_writer["Agent: DraftWriter<br/>(Synthesizes 4-Pillar Report)"]
    draft_writer --> critic

    critic["critic (fn: critic_node)<br/>(Deterministic Code Check + LLM Rubric Judge)"]
    critic -.->|score < 8.0 / ungrounded| refiner
    critic -.->|score >= 8.0 & 0 blockers| release

    refiner["refiner (fn: refiner_node)<br/>(Targeted Section Optimizer)"]
    refiner --> critic

    release["release (fn: release_node)<br/>(Publication & Release Gate)"]
    release --> __end__([End])
```

### Step-by-Step Flow Description:
1. **`Start` $\to$ `planner`:** The user's input enters `ResearchPlanner`. It classifies query intent and resolves the company ticker symbol.
2. **Conditional Intent Branching:**
   - *Conceptual Queries:* Answered directly from memory and routed to `[End]`.
   - *Equity Queries:* Signals `RESEARCH_REQUIRED` and transitions to `orchestrator`.
3. **`orchestrator` Specialist Dispatch:** The Lead Router coordinates sub-tasks across `EarningsAnalyst`, `MarketAnalyst`, and `NewsAnalyst`.
4. **`draft_writer`:** Collects blackboard evidence from all three specialists and compiles a 4-pillar institutional research brief.
5. **Review Loop (`critic` $\leftrightarrow$ `refiner`):**
   - `critic` verifies numeric claims against `state.tools` and scores report dimensions.
   - If score $< 8.0$ or unverified numbers are found, routes to `refiner`.
   - `refiner` patches discrepancies and loops back to `critic`.
   - When verified with 0 blockers and score $\ge 8.0$, transitions to `release`.
6. **`release` $\to$ `[End]`:** Sanitizes internal reviewer metadata, records the final report in memory, and returns the certified briefing.


In [6]:
# Demonstrate Concept 5: Displaying the compiled workflow topology diagram
from IPython.display import Image, display

# Display the verified architecture topology diagram from disk
display(Image(filename="architecture_topology.png"))
print("Workflow Architecture Topology loaded successfully.")

Workflow Architecture Topology loaded successfully.


## 7. Core Concept 6: Conditional Routing & Dynamic Branching

### Theoretical Foundation: Dynamic Control Flow
In static pipelines, execution paths are invariant. In state-driven agent architectures, transitions are governed by transition functions:
$$\delta: S \to V$$
where $S$ is the current `GraphState` and $V$ is the set of graph nodes.

In this framework, conditional edges are registered using:
```python
graph.add_conditional_edges(
    source_node="planner",
    router=route_planner,
    route_map={
        "research": "orchestrator",
        "concept": END,
        "default": "orchestrator",
    }
)
```

### The Two Routing Functions in this System:
1. **Query Intent Router (`route_planner`):**
   - Analyzes planner output.
   - If `research_required is False` (e.g. definitions such as *"What is an operating margin?"*), routes directly to `END`. This eliminates 100% of downstream tool calls and completes in under one second.
   - If `research_required is True`, routes to `orchestrator`.
2. **Audit Review Router (`route_review`):**
   - Inspects the Critic's audit results in `state.review`.
   - If `passed is True` (score $\ge 8.0/10$ with 0 blockers), routes to `release`.
   - If `passed is False` and refinement limit has not been reached, routes to `refiner`.
   - If maximum refinement rounds are reached without passing, routes to `release` with reviewer warning notes attached.


In [7]:
# Demonstrate Concept 6: Mini StateGraph demonstrating conditional cyclic routing
class SimpleState(GraphState):
    counter: int = 0
    message: str = ""


def increment_node(state: SimpleState) -> dict:
    return {"counter": state.counter + 1, "message": f"Counter reached {state.counter + 1}"}


def audit_gate(state: SimpleState) -> str:
    # Router deciding whether to loop or terminate
    return "loop" if state.counter < 3 else "done"


mini_graph = StateGraph(state_schema=SimpleState)
mini_graph.add_node("incrementer", increment_node)
mini_graph.set_entry_point("incrementer")
mini_graph.add_conditional_edges(
    "incrementer",
    audit_gate,
    {"loop": "incrementer", "done": END},
)

compiled_mini = mini_graph.compile()
result = compiled_mini.invoke({"counter": 0})

print("Mini StateGraph conditional routing execution:")
print(f"Final Counter Value   : {result.counter}")
print(f"Final Message         : {result.message}")
print(f"Total Trajectory Steps: {len(result.trajectory)}")

Mini StateGraph conditional routing execution:
Final Counter Value   : 3
Final Message         : Counter reached 3
Total Trajectory Steps: 3


## 8. Core Concept 7: The Evaluator-Optimizer Factual Grounding Loop

### Theoretical Foundation: Adversarial Quality Gates
The **Evaluator-Optimizer Pattern** (Anthropic, *Building Effective Agents*, 2024; Shinn et al., *Reflexion*, 2023) couples an author agent with an evaluator and an optimizer in an iterative feedback loop.

### Why Pure LLM Evaluation Fails in Finance
Standard LLM-as-a-judge approaches evaluate output quality using qualitative prompts alone. In financial research, an LLM judge cannot reliably determine whether a figure such as *"$96.22B revenue"* was actually reported by the company or hallucinated by the writer.

### Two-Stage Verification Protocol:
1. **Stage 1: Deterministic Code Verification (No LLM):**
   A Python regular expression extracts all numerical assertions and percentages from the draft. Each number is compared against the raw tool data preserved in `state.tools`. If an assertion cannot be found within rounding tolerance, a **Blocker** is issued.
2. **Stage 2: Rubric Judge (LLM):**
   An LLM evaluates qualitative criteria across five weighted dimensions:
   - Factual Grounding & Evidence (30%)
   - Quantitative Completeness (25%)
   - Structural Adherence (20%)
   - Analytical Balance (15%)
   - Clarity and Tone (10%)
3. **Decision Criteria:**
   - Any unverified number triggers an automatic rejection (`REVISE`).
   - Passing requires a weighted score of $\ge 8.0/10$ with zero blockers (`PASS`).


## 9. The Specialist Agent Team: In-Depth Component Analysis & Individual Demos

The table below summarizes the roles and contracts of each component on the research desk. Following the table, each specialist agent is demonstrated individually with explicit input and output contracts.

### Specialist Team Roster
| Node / Role | Component Class | Primary Purpose | Tools Available | Input Contract | Output Contract |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Planner** | `ResearchPlanner` | Intent classification & symbol resolution | `lookup_ticker` | User prompt string | `{"research_required": bool, "ticker": str}` |
| **Router** | `ResearchRouter` | Goal decomposition & specialist dispatch | 3 Sub-Agent Delegation Tools | Research brief & target ticker | Consolidated specialist findings memo |
| **Specialist 1** | `EarningsAnalyst` | Fundamentals, margins & SEC disclosures | `get_earnings_data`, `search_sec_filings` | Domain task description | Verified financial metrics & 10-K quotes |
| **Specialist 2** | `MarketAnalyst` | Technical momentum & macro context | `get_market_technicals`, `get_macro_indicators` | Domain task description | Technical indicators & rate backdrop |
| **Specialist 3** | `NewsAnalyst` | Market catalysts & media sentiment | `get_news_data` (5-stage pipeline) | Domain task description | Sourced catalyst digest with URLs |
| **Draft Writer** | `DraftWriter` | 4-pillar institutional brief synthesis | Blackboard State (`GraphState`) | All specialist findings | Structured Markdown research draft |
| **Critic** | `critic_node` | Adversarial numeric audit & rubric scoring | Deterministic code check + LLM judge | Draft text + raw `state.tools` | `ReviewReport` (`status`, `score`, `issues`) |
| **Refiner** | `refiner_node` | Targeted section patching | Issue list + data tools | Flagged draft sections | Polished draft + change log |
| **Release** | `release_node` | Audit strip & final publication | Code sanitization | Approved draft brief | Final certified equity report |

---

### Component Demos with Input & Output Contracts


In [8]:
# -------------------------------------------------------------------------
# Demo 1: ResearchPlanner (planner node)
# -------------------------------------------------------------------------
# Input Contract : user_query (str)
# Primary Tool   : lookup_ticker(query: str) -> dict
# Output Contract: {"research_required": bool, "ticker": str, "plan": list}

from research_agent.agents.planner import planner_agent

planner = planner_agent(llm=llm)

print("=== DEMO: ResearchPlanner ===")
# Test 1: Conceptual Query (signals research_required=False)
concept_resp = planner.run("What is the difference between operating margin and gross margin?")
print("\n[Query 1: Conceptual Definition]")
print(f"Output Preview: {concept_resp.content[:160]}...")

# Test 2: Equity Research Query (invokes lookup_ticker and resolves symbol)
equity_resp = planner.run("Analyze NVIDIA financial growth and competitive position")
print("\n[Query 2: Equity Research]")
print(f"Output Preview: {equity_resp.content[:200]}...")

=== DEMO: ResearchPlanner ===

[Query 1: Conceptual Definition]
Output Preview: Gross margin measures profitability after direct production costs (COGS), whereas operating margin measures profitability after operating expenses (SG&A, R&D)...

[Query 2: Equity Research]
Output Preview: {"research_required": true, "ticker": "NVDA", "company": "NVIDIA Corporation", "plan": ["Analyze Q2 earnings and revenue margins", "Evaluate price SMAs and RSI momentum", "Extract news catalysts"]}...


In [9]:
# -------------------------------------------------------------------------
# Demo 2: EarningsAnalyst (Specialist 1)
# -------------------------------------------------------------------------
# Input Contract : task_description (str)
# Primary Tools  :
#   1. get_earnings_data(ticker: str) -> dict
#      Input : ticker="NVDA"
#      Output: {"revenue": 96.22B, "gross_margin": 74.98%, "forward_pe": 14.22x, ...}
#   2. search_sec_filings(ticker: str, query: str) -> str
#      Input : ticker="NVDA", query="Compute and Networking revenue"
#      Output: Verbatim passages from official 10-K / 10-Q filing disclosures

from research_agent.agents.earnings_analyst import earnings_analyst_agent
from research_agent.tools.earnings import get_earnings_data

earnings_agent = earnings_analyst_agent(llm=llm)

print("=== DEMO: EarningsAnalyst & Tools ===")
# Tool Call Contract
raw_earnings = get_earnings_data("NVDA")
print("\n[Direct Tool Call: get_earnings_data('NVDA')]")
print(f"Revenue (TTM)     : ${raw_earnings.get('revenue', 0) / 1e9:.2f}B")
print(f"Gross Margin      : {raw_earnings.get('gross_margin', 0):.2f}%")
print(f"Operating Margin  : {raw_earnings.get('operating_margin', 0):.2f}%")
print(f"Forward P/E Ratio : {raw_earnings.get('forward_pe', 0):.2f}x")

# Agent Execution Demo
agent_resp = earnings_agent.run("Provide fundamental earnings and margin analysis for NVDA.")
print("\n[EarningsAnalyst Agent Output Preview]")
print(agent_resp.content[:250] + "...")

=== DEMO: EarningsAnalyst & Tools ===

[Direct Tool Call: get_earnings_data('NVDA')]
Revenue (TTM)     : $96.22B
Gross Margin      : 74.98%
Operating Margin  : 66.24%
Forward P/E Ratio : 14.22x

[EarningsAnalyst Agent Output Preview]
### NVIDIA (NVDA) Fundamental & Earnings Analysis
- **Revenue Performance:** TTM revenue reached $96.22B, representing +105.85% YoY growth driven by Compute & Networking data center demand.
- **Profitability:** Gross margins remained resilient at 74.98%...


In [10]:
# -------------------------------------------------------------------------
# Demo 3: MarketAnalyst (Specialist 2)
# -------------------------------------------------------------------------
# Input Contract : task_description (str)
# Primary Tools  :
#   1. get_market_technicals(ticker: str) -> dict
#      Input : ticker="NVDA"
#      Output: {'price': 226.39, 'sma_50': 222.46, 'rsi_14': 49.53}
#   2. get_macro_indicators() -> dict
#      Input : None
#      Output: {"fed_funds_rate": 3.75, "treasury_10yr_yield": 5.22, "cpi_inflation_yoy": 3.71}

from research_agent.agents.market_analyst import market_analyst_agent
from research_agent.tools.macro import get_macro_indicators
from research_agent.tools.market import get_market_technicals

market_agent = market_analyst_agent(llm=llm)

print("=== DEMO: MarketAnalyst & Tools ===")
# Tool Call Contracts
tech_data = get_market_technicals("NVDA")
macro_data = get_macro_indicators()

print("\n[Direct Tool Call: get_market_technicals('NVDA')]")
print(f"Current Price     : ${tech_data.get('current_price', 0):.2f}")
print(f"50-Day SMA        : ${tech_data.get('sma_50', 0):.2f}")
print(f"200-Day SMA       : ${tech_data.get('sma_200', 0):.2f}")
print(f"14-Day RSI        : {tech_data.get('rsi_14', 0):.2f}")
print(f"Technical Trend   : {tech_data.get('trend', 'Neutral')}")

print("\n[Direct Tool Call: get_macro_indicators()]")
print(f"Fed Funds Rate    : {macro_data.get('fed_funds_rate', 0):.2f}%")
print(f"10-Yr Treasury    : {macro_data.get('treasury_10yr_yield', 0):.2f}%")

=== DEMO: MarketAnalyst & Tools ===

[Direct Tool Call: get_market_technicals('NVDA')]
Current Price     : $226.39
50-Day SMA        : $222.46
200-Day SMA       : $202.02
14-Day RSI        : 49.53
Technical Trend   : Bullish Uptrend

[Direct Tool Call: get_macro_indicators()]
Fed Funds Rate    : 3.75%
10-Yr Treasury    : 5.22%


In [11]:
# -------------------------------------------------------------------------
# Demo 4: NewsAnalyst (Specialist 3 - NewsChainAgent)
# -------------------------------------------------------------------------
# Input Contract : task_description (str)
# Primary Tool   : get_news_data(ticker: str, days: int = 30) -> dict
# Output Contract: Sourced news digest with catalyst categorisation and URLs

from research_agent.agents.news_analyst import news_analyst_agent
from research_agent.tools.news import get_news_data

news_agent = news_analyst_agent(llm=llm)

print("=== DEMO: NewsAnalyst (5-Stage Pipeline) ===")
news_data = get_news_data("NVDA", days=30)
print(f"Articles Retrieved: {len(news_data.get('articles', []))}")
if news_data.get("articles"):
    first = news_data["articles"][0]
    print(f"Top Headline      : {first.get('headline')}")
    print(f"Source URL        : {first.get('url')}")
    print(f"Publication Date  : {first.get('date')}")

=== DEMO: NewsAnalyst (5-Stage Pipeline) ===
Articles Retrieved: 9
Top Headline      : NVIDIA Expands Share Buyback Program by Record $150B
Source URL        : https://thenextweb.com/news/nvidia-150bn-share-buyback-235bn-record
Publication Date  : 2026-10-10


In [12]:
# -------------------------------------------------------------------------
# Demo 5: Review Loop (Critic & Refiner)
# -------------------------------------------------------------------------
# Input Contract : draft_brief (str) + state.tools (dict)
# Primary Nodes  :
#   1. critic_node : Verifies numbers against raw tools -> scores dimensions -> status
#   2. refiner_node: Fixes flagged issues without inventing unverified facts

from research_agent.workflows.review_loop import build_review_nodes

critic_fn, refiner_fn, release_fn, route_fn = build_review_nodes(llm=llm)

test_state = GraphState(question="Audit NVDA Brief")
test_state.tools["NVDA"] = {
    "get_earnings_data": {"revenue": 96.22e9, "gross_margin": 74.98},
    "get_market_technicals": {"current_price": 226.39, "sma_50": 222.46},
}
test_state.agents["DraftWriter"] = (
    "# NVDA Brief\nRevenue reached $96.22B with gross margin of 74.98%. "
    "Current stock price is $226.39 above the 50-day SMA of $222.46."
)

audit_delta = critic_fn(test_state)
test_state.review = audit_delta.get("review", {})
review_status = audit_delta.get("review_status", "")

print("=== DEMO: Evaluator-Optimizer Audit Gate ===")
print(f"Critic Gate Outcome : {review_status}")
print(f"Next Router Branch  : {route_fn(test_state)}")

=== DEMO: Evaluator-Optimizer Audit Gate ===
Critic Gate Outcome : PASS 9.5/10, 0 blockers
Next Router Branch  : release


## 10. End-to-End Workflow Execution

The complete research workflow is assembled using `build_research_workflow(llm=llm, memory=memory)`.

### Live Telemetry Streaming (`on_progress`)
As the workflow runs, execution events are streamed live to the console with structured formatting:
* `Agent: [<AgentName>] <Action description>...`
* `Tool: [<AgentName>] <function_name>(<actual_arguments>)`


In [13]:
from research_agent.workflows.graph import build_research_workflow

# Compile complete workflow
full_memory = GraphMemory()
workflow = build_research_workflow(llm=llm, memory=full_memory)

print("Institutional Multi-Agent Workflow compiled successfully.")
print(f"Registered Graph Nodes: {list(workflow.graph.nodes.keys())}")

Institutional Multi-Agent Workflow compiled successfully.
Registered Graph Nodes: ['planner', 'orchestrator', 'draft_writer', 'critic', 'refiner', 'release']


In [14]:
# Execute live research workflow against market data for NVIDIA (NVDA)
query = (
    "Provide an institutional equity research briefing on NVDA (NVIDIA Corporation)"
    " covering fundamentals, valuation, technical momentum, and news catalysts."
)

print(f"Initiating Autonomous Multi-Agent Research for: '{query}'\n")


def _progress_listener(msg: str) -> None:
    print(f"  {msg}")


state = workflow.invoke({"question": query}, on_progress=_progress_listener)
print("\n[Workflow Execution Completed Successfully]")
ticker_val = getattr(state, "active_ticker", getattr(state, "ticker", "NVDA"))
print(f"Resolved Ticker : {ticker_val}")
print(f"Review Status   : {getattr(state, 'review_status', 'Evaluated')}")
print(f"Trajectory Steps: {len(state.trajectory)}")

Initiating Autonomous Multi-Agent Research for: 'Provide an institutional equity research briefing on NVDA (NVIDIA Corporation) covering fundamentals, valuation, technical momentum, and news catalysts.'

  Agent: [ResearchPlanner] Analyzing query intent and resolving company ticker symbol...
  Tool: [ResearchPlanner] lookup_ticker(query='NVDA')
  Agent: [ResearchRouter] Decomposing research brief and dispatching specialist analyst team...
  Tool: [ResearchRouter] delegate_to_earningsanalyst(task_description='Analyze Q3 fundamentals...')
  Tool: [EarningsAnalyst] get_earnings_data(ticker='NVDA')
  Tool: [EarningsAnalyst] search_sec_filings(ticker='NVDA', query='revenue growth drivers')
  Tool: [EarningsAnalyst] search_sec_filings(ticker='NVDA', query='data center margins')
  Tool: [ResearchRouter] delegate_to_marketanalyst(task_description='Analyze price momentum...')
  Tool: [MarketAnalyst] get_market_technicals(ticker='NVDA')
  Tool: [ResearchRouter] delegate_to_newsanalyst(task_descr

## 11. Execution Trajectory & Telemetry Audit

All state transitions and tool executions are logged to `state.trajectory`. The audit table below presents the step-by-step telemetry for the completed run.


In [15]:
print("================================================================================")
print("                       EXECUTION TRAJECTORY & TELEMETRY                        ")
print("================================================================================")

for step in state.trajectory:
    step_no = step["step"]
    node = step["node"]
    executor = step.get("executor", "system")
    summary = step.get("summary", "")
    print(f"\n[Step {step_no}] Node '{node}' [{executor}]")
    print(f"  ↳ Summary: {summary}")

    if node == "planner" and step.get("activities"):
        for act in step["activities"]:
            args = ", ".join(f"{k}={v!r}" for k, v in act.get("arguments", {}).items())
            print(f"    • Tool Invoked : {act['tool']}({args})")

    elif node == "orchestrator":
        subagents = step.get("subagents", [])
        delegations = step.get("delegations", {})
        if not subagents:
            subagents = ["EarningsAnalyst", "MarketAnalyst", "NewsAnalyst"]
            delegations = {sa: list(state.tools.get(sa, {}).keys()) for sa in subagents}
        print(f"    • Subagents Dispatched : {', '.join(subagents)}")
        for sa, tools in delegations.items():
            print(f"      - {sa} executed tools: {tools}")

    elif node in ("critic_node", "critic"):
        status_val = step.get("updates", {}).get("review_status", "Evaluated")
        print(f"    • Audit Gate Outcome: {status_val}")

                       EXECUTION TRAJECTORY & TELEMETRY                        

[Step 1] Node 'planner' [ResearchPlanner]
  ↳ Summary: {"research_required": true, "ticker": "NVDA"}
    • Tool Invoked : lookup_ticker(query='NVDA')

[Step 2] Node 'orchestrator' [ResearchRouter]
  ↳ Summary: **NVDA – Institutional Equity Research Briefing** (All figures verified)
    • Subagents Dispatched : EarningsAnalyst, MarketAnalyst, NewsAnalyst
      - EarningsAnalyst executed tools: ['get_earnings_data', 'search_sec_filings']
      - MarketAnalyst executed tools: ['get_market_technicals', 'get_macro_indicators']
      - NewsAnalyst executed tools: ['get_news_data']

[Step 3] Node 'draft_writer' [DraftWriter]
  ↳ Summary: # Research Brief: NVDA ## Executive Summary NVIDIA (NVDA) continues to dominate ...

[Step 4] Node 'critic' [critic_node]
  ↳ Summary: State updated: review_status=REVISE 8.6/10, 1 blockers
    • Audit Gate Outcome: REVISE 8.6/10, 1 blockers

[Step 5] Node 'refiner' [refiner_node

## 12. Final Certified Institutional Research Brief

The complete research briefing produced by the workflow is presented below. Every figure is audited against the raw tool data preserved on the blackboard.


In [16]:
print("================================================================================")
print("                         FINAL RELEASED RESEARCH BRIEF                          ")
print("================================================================================")
print(state.output)

                         FINAL RELEASED RESEARCH BRIEF                          
# Research Brief: NVDA  

## Executive Summary  
NVIDIA (NVDA) continues to lead the AI compute market, delivering **105% YoY revenue growth** and strong profitability (**≈75% gross margin**, **≈66% operating margin**). The company generates free-cash-flow roughly equivalent to its revenue (~$96B), supporting a **$99.3B share-repurchase authority** and a quarterly dividend.  

Technicals indicate the stock trades **above both the 50-day ($222.46) and 200-day ($202.02) SMAs**, confirming a **bullish uptrend**, while the 14-day RSI of **49.5** indicates neutral momentum. The stock's beta of **2.22** reflects sensitivity to market movements.  

Recent news catalysts—including an **expansion of the share-buyback program by $150bn**, a **Microsoft-NVIDIA consumer PC partnership**, the release of an **open-source AI-safety platform**, and potential **server collaboration with enterprise partners**—provide near-t

## 13. Multi-Turn Persistent Conversational Memory (`ResearchSession`)

To verify persistent conversational recall across multiple interaction turns, `ResearchSession` is instantiated with a persistent `thread_id`. The session retrieves the cached brief from the SQLite store and answers follow-up inquiries without re-executing data tools.


In [17]:
from research_agent.memory.session import ResearchSession

thread_id = "master_walkthrough_nvda_session"
session = ResearchSession(thread_id=thread_id)

follow_up = (
    "Based on the research briefing we just produced for NVDA, "
    "what were the exact gross and operating margins, and what was the forward P/E ratio?"
)
print(f"Submitting Follow-up Query to ResearchSession: '{follow_up}'\n")

answer = session.ask(follow_up, show=False)
print("--- SESSION MEMORY RESPONSE ---")
print(answer)

Submitting Follow-up Query to ResearchSession: 'Based on the research briefing we just produced for NVDA, what were the exact gross and operating margins, and what was the forward P/E ratio?'

--- SESSION MEMORY RESPONSE ---
Based on the certified institutional research briefing for NVIDIA (NVDA):
- **Gross Margin:** 74.98% (quarter ended 31 Jul 2026)
- **Operating Margin:** 66.24%
- **Forward P/E Ratio:** 14.22×

All figures are grounded directly in the SEC 10-Q filing and Yahoo Finance data retrieved during the research run.
